# Урок 6 · Зір для тих, хто погано бачить
**Як бачить машина · 9-А · робочий зошит**

**Спершу:** *Файл → Зберегти копію на Диску*. Без цього ваші зміни зникнуть після уроку.

**Як працювати**
* Кроки йдуть так само, як на слайдах: слайд «Зошит · крок 2» = розділ «Крок 2» тут.
* ▶ ліворуч від клітинки — запустити її. Клітинки запускаємо **згори вниз**.
* `___` — пропуск: його заповнюєте ви.
* 💡 **Підказка** — клітинка з прихованим кодом. Відкривайте, лише якщо застрягли.
* **Крок 0 запускайте одразу** — він ставить модулі й качає «знання букв», це хвилина-дві.

In [ ]:
#@title Службове: цінник, показ і гістограма (можна не відкривати) { display-mode: "form" }
import os
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw, ImageFont

FONT = os.path.join(matplotlib.get_data_path(), 'fonts/ttf/DejaVuSans-Bold.ttf')

def tag(shadow=False):
    """Тестове фото цінника 640×320. shadow=True — праворуч на аркуш лягла густа тінь."""
    im = Image.new('L', (640, 320), 255)
    d = ImageDraw.Draw(im)
    d.text((320, 105), 'ХЛІБ', font=ImageFont.truetype(FONT, 110), fill=0, anchor='mm')
    d.text((320, 240), '32 грн', font=ImageFont.truetype(FONT, 80), fill=0, anchor='mm')
    ink = np.array(im) < 128                         # де чорнило
    f = np.linspace(0, 1, 640)[None, :] * np.ones((320, 1))
    if shadow:
        paper = 235 - 190 * f ** 1.4                 # папір: 235 зліва → 45 у тіні
        dark = 30 - 18 * f                           # чорнило: 30 → 12
    else:
        paper = 235 - 20 * f
        dark = 30 + 0 * f
    return np.where(ink, dark, paper).astype(np.uint8)

def show(*imgs, titles=None):
    """Показати кілька сірих фото поруч."""
    fig, ax = plt.subplots(1, len(imgs), figsize=(5 * len(imgs), 2.6))
    ax = np.atleast_1d(ax)
    for i, im in enumerate(imgs):
        ax[i].imshow(im, cmap='gray', vmin=0, vmax=255)
        ax[i].axis('off')
        if titles: ax[i].set_title(titles[i])
    plt.show()

def bars(im, title=''):
    """Гістограма одного фото (як на уроці 5)."""
    h, _ = np.histogram(im, bins=256, range=(0, 256))
    plt.figure(figsize=(7, 2.4))
    plt.bar(range(256), h, width=1)
    plt.xlim(0, 255); plt.xticks(range(0, 256, 16)); plt.title(title)
    plt.show()


## Крок 0 · Ставимо модулі й беремо цінник
`easyocr` — розпізнає текст на фото, `gTTS` — перетворює текст на голос.
Запустіть і не чекайте: поки качається, слухаємо пояснення.

In [ ]:
!pip install -q easyocr gTTS            # два модулі: розпізнавання й озвучення (~1 хв)
import easyocr
reader = easyocr.Reader(['uk', 'en'])   # «знання букв» — завантажиться один раз
img = tag()                             # тестове фото цінника
show(img)
print(img.shape, img.min(), img.max())

## Крок 1 · Машина читає цінник
`reader.readtext` шукає на фото рядки тексту й повертає їх списком. `detail=0` — лише текст, без координат.

In [ ]:
words = reader.readtext(img, detail=0)   # список рядків, які машина знайшла
print(words)
text = ' '.join(words)
print(text)

✍️ **Ваша відповідь:** скільки рядків знайшла машина? Чи все прочитано правильно?

*(пишіть тут)*

## Крок 2 · Машина говорить
`gTTS` надсилає текст сервісу Google і отримує звук. Заповніть пропуск `___` — код української мови — і натисніть ▶ у програвачі.

In [ ]:
from gtts import gTTS
from IPython.display import Audio
gTTS(text, lang='___').save('say.mp3')   # мова — українська
Audio('say.mp3')

In [ ]:
#@title 💡 Підказка { display-mode: "form" }
# Код української мови — 'uk'
# gTTS(text, lang='uk').save('say.mp3')

## Крок 3 · Тінь і поріг
На цінник лягла густа тінь. Спершу подивіться, що вийде, і на гістограму.

In [ ]:
dim = tag(shadow=True)                  # той самий цінник, але праворуч густа тінь
show(dim)
print(reader.readtext(dim, detail=0))
bars(dim, 'гістограма фото з тінню')

На гістограмі дві купки: ліворуч — чорнило, праворуч — папір.
**Поріг** — число між ними: світліше за нього стає білим, темніше — чорним.
Замініть `___` на свій поріг і порівняйте з «порогом навмання» 128.

In [ ]:
t = ___                                 # поріг: число МІЖ двома купками гістограми
b = np.where(dim > t, 255, 0).astype(np.uint8)   # світліше за t — біле, решта — чорне
b128 = np.where(dim > 128, 255, 0).astype(np.uint8)
show(dim, b128, b, titles=['з тінню', 'поріг 128', 'поріг ' + str(t)])
print('128:', reader.readtext(b128, detail=0))
print(t, ':', reader.readtext(b, detail=0))

In [ ]:
#@title 💡 Підказка { display-mode: "form" }
# Ліва купка — чорнило (до 25), права — папір (від 45).
# Між ними порожньо: підходить будь-яке число від 25 до 44.
# t = 35

✍️ **Ваша відповідь:** який поріг ви обрали і чому? Що сталося з буквами при порозі 128?

*(пишіть тут)*

## Крок 4 · Своє фото *(для охочих)*
Сфотографуйте телефоном будь-який текст (обкладинку, вивіску, цінник).
Панель 📁 ліворуч → завантажити файл → назвати **foto.jpg**. Запустіть — Colab прочитає його вголос.

In [ ]:
from PIL import Image
own = np.array(Image.open('foto.jpg').convert('L'))
show(own)
own_text = ' '.join(reader.readtext(own, detail=0))
print(own_text)
gTTS(own_text, lang='uk').save('own.mp3')
Audio('own.mp3')

## ДЗ · Доступність на вашому телефоні
Знайдіть і спробуйте три функції:
1. **лупу**;
2. **озвучення екрана** (TalkBack / «Вибрати й прослухати» на Android, «Промовляння» на iPhone);
3. **розпізнавання тексту камерою** (Lookout або Google Lens → «Текст» → «Слухати» на Android, «Текст з камери» на iPhone).

✍️ **ДЗ:** як називається кожна функція на вашому телефоні? Яка з них корисна саме вам — і чому?

1. …
2. …
3. …

*(пишіть тут)*